In [1]:
import numpy as np
import collections
import urllib.request
import zipfile
import os

### Data Preprocessing & Vocabulary Building

In [2]:
def download_data():
    url = 'http://mattmahoney.net/dc/text8.zip'
    filename = 'data/text8.zip'
    if not os.path.exists(filename):
        print("Downloading text8 dataset...")
        urllib.request.urlretrieve(url, filename)
    
    with zipfile.ZipFile(filename) as f:
        # text8 is just one giant line of words separated by spaces
        return f.read(f.namelist()[0]).decode('utf-8').split()

# 1. Load and Tokenize
words = download_data()
print(f"Total words in corpus: {len(words)}")

# 2. Build Vocabulary
# We limit vocab to top 10,000 words for a faster demo, 
# but you can increase this for better results.
VOCAB_SIZE = 10000
count = [['UNK', -1]]
count.extend(collections.Counter(words).most_common(VOCAB_SIZE - 1))

word_to_id = {word: i for i, (word, _) in enumerate(count)}
id_to_word = {i: word for word, i in word_to_id.items()}

# Convert the whole corpus to IDs (words not in vocab become 0/UNK)
data = [word_to_id.get(w, 0) for w in words]

# 3. Subsampling
def subsample(data, word_to_id, id_to_word):
    counts = collections.Counter(data)
    total_count = len(data)
    threshold = 1e-3
    
    # Calculate keep probabilities
    probs = {}
    for idx, count in counts.items():
        f = count / total_count
        # The Mikolov formula for keep probability
        p_keep = (np.sqrt(f / threshold) + 1) * (threshold / f)
        probs[idx] = p_keep
        
    # Filter data: only keep word if random.uniform < p_keep
    train_data = [w for w in data if np.random.random() < probs.get(w, 0)]
    return train_data

train_data = subsample(data, word_to_id, id_to_word)
print(f"Words after subsampling: {len(train_data)}")

Total words in corpus: 17005207
Words after subsampling: 11268771


### Build The Negative Sampling Table

In [3]:
def create_unigram_table(counts, vocab_size, table_size=100_000_000):
    # 1. Get frequencies and raise to 3/4 power
    # counts is from collections.Counter(data)
    f_pow = np.array([counts.get(i, 0) for i in range(vocab_size)])**0.75
    
    # 2. Normalize to create a probability distribution
    probs = f_pow / np.sum(f_pow)
    
    # 3. Fill the table: each word ID appears in proportion to its probability
    # We use round to determine how many slots each word gets
    num_slots = np.round(probs * table_size).astype(int)
    
    # Create the table as a giant 1D array
    table = []
    for word_id, count in enumerate(num_slots):
        table.extend([word_id] * count)
    
    return np.array(table)

# Execute for our vocab
# Note: we use the counts from the ORIGINAL data to get true distribution
unigram_table = create_unigram_table(collections.Counter(data), VOCAB_SIZE)
print(f"Unigram table created with {len(unigram_table)} entries.")

Unigram table created with 100000135 entries.


### Matrix Initialization

In [ ]:
EMBEDDING_DIM = 100  # Each word is represented by a 100-element vector

def initialize_weights(vocab_size, embedding_dim):
    # We use a uniform distribution centered around zero.
    # The scale is often 0.5 / embedding_dim as a heuristic.
    
    W_in = np.random.uniform(
        low=-0.5/embedding_dim, 
        high=0.5/embedding_dim, 
        size=(vocab_size, embedding_dim)
    )
    
    W_out = np.random.uniform(
        low=-0.5/embedding_dim, 
        high=0.5/embedding_dim, 
        size=(vocab_size, embedding_dim)
    )
    
    return W_in, W_out

# Initialize our "Brain"
W_in, W_out = initialize_weights(VOCAB_SIZE, EMBEDDING_DIM)

print(f"Target Matrix Shape: {W_in.shape}")
print(f"Context Matrix Shape: {W_out.shape}")

Target Matrix Shape: (10000, 100)
Context Matrix Shape: (10000, 100)


### The Training Loop

In [5]:
def sigmoid(x):
    # Clips values to avoid overflow in exp
    return 1 / (1 + np.exp(-np.clip(x, -10, 10)))

# Hyperparameters
WINDOW_SIZE = 2
K_NEG = 5           # Number of negative samples per positive pair
LEARNING_RATE = 0.01
EPOCHS = 1          # For text8, 1 epoch is usually enough for a demo

for epoch in range(EPOCHS):
    total_loss = 0
    
    # We iterate through the data with a sliding window
    for i, target_id in enumerate(train_data):
        # 1. Define the context window
        start = max(0, i - WINDOW_SIZE)
        end = min(len(train_data), i + WINDOW_SIZE + 1)
        
        # 2. Get the Target Vector (from W_in)
        v_t = W_in[target_id] # Shape: (100,)
        
        # 3. Process each Context Word in the window
        for j in range(start, end):
            if i == j: continue # Skip the target word itself
            
            context_id = train_data[j]
            
            # --- POSITIVE SAMPLE ---
            u_pos = W_out[context_id]
            z_pos = np.dot(v_t, u_pos)
            y_hat_pos = sigmoid(z_pos)
            
            err_pos = y_hat_pos - 1
            # Loss for logging: -log(sigmoid(z))
            total_loss -= np.log(y_hat_pos + 1e-9)
            
            # --- NEGATIVE SAMPLES ---
            # Pick K random indices from our pre-built unigram table
            neg_indices = np.random.choice(unigram_table, size=K_NEG)
            u_negs = W_out[neg_indices] # Shape: (K, 100)
            
            # Vectorized dot product for all negative samples
            z_negs = np.dot(u_negs, v_t) # Shape: (K,)
            y_hat_negs = sigmoid(z_negs)
            
            err_negs = y_hat_negs - 0
            total_loss -= np.sum(np.log(1 - y_hat_negs + 1e-9))
            
            # --- GRADIENTS & UPDATES ---
            # Calculate gradient for the target vector v_t
            # grad_v_t = (err_pos * u_pos) + sum(err_neg * u_neg)
            grad_v_t = (err_pos * u_pos) + np.dot(err_negs, u_negs)
            
            # Update Context Vectors (u_pos and u_negs)
            W_out[context_id] -= LEARNING_RATE * (err_pos * v_t)
            W_out[neg_indices] -= LEARNING_RATE * np.outer(err_negs, v_t)
            
            # Update Target Vector (v_t)
            W_in[target_id] -= LEARNING_RATE * grad_v_t

        if i % 100000 == 0:
            print(f"Progress: {i}/{len(train_data)}, Loss: {total_loss/100000:.4f}")
            total_loss = 0

Progress: 0/11268771, Loss: 0.0001
Progress: 100000/11268771, Loss: 15.4108
Progress: 200000/11268771, Loss: 12.6080
Progress: 300000/11268771, Loss: 11.3906
Progress: 400000/11268771, Loss: 10.8211
Progress: 500000/11268771, Loss: 10.5442
Progress: 600000/11268771, Loss: 10.3142
Progress: 700000/11268771, Loss: 9.8497
Progress: 800000/11268771, Loss: 9.9054
Progress: 900000/11268771, Loss: 10.0296
Progress: 1000000/11268771, Loss: 9.9064
Progress: 1100000/11268771, Loss: 9.9249
Progress: 1200000/11268771, Loss: 9.8965
Progress: 1300000/11268771, Loss: 9.8604
Progress: 1400000/11268771, Loss: 9.8634
Progress: 1500000/11268771, Loss: 9.8369
Progress: 1600000/11268771, Loss: 9.6769
Progress: 1700000/11268771, Loss: 9.5690
Progress: 1800000/11268771, Loss: 9.5997
Progress: 1900000/11268771, Loss: 9.6811
Progress: 2000000/11268771, Loss: 9.6353
Progress: 2100000/11268771, Loss: 9.5902
Progress: 2200000/11268771, Loss: 9.4553
Progress: 2300000/11268771, Loss: 9.7231
Progress: 2400000/112687

### Evaluation (Inference)

In [ ]:
def get_similarity(word, word_to_id, W_in, top_n=8):
    if word not in word_to_id:
        return "Word not in vocabulary."
    
    word_id = word_to_id[word]
    v_t = W_in[word_id]
    
    # 1. Normalize the Target Vector
    v_t_norm = v_t / np.linalg.norm(v_t)
    
    # 2. Normalize the entire Matrix (all words)
    # We add a tiny epsilon to avoid division by zero
    W_norms = np.linalg.norm(W_in, axis=1, keepdims=True) + 1e-9
    W_normalized = W_in / W_norms
    
    # 3. Calculate Dot Product (Cosine Similarity)
    # (1, 100) @ (100, 10000) -> (10000,)
    similarities = np.dot(W_normalized, v_t_norm)
    
    # 4. Get Top N (excluding the word itself, which will be index 0)
    closest_ids = np.argsort(similarities)[::-1][1:top_n+1]
    
    return [(id_to_word[idx], similarities[idx]) for idx in closest_ids]

# Example usage:
print("Similar to 'king':", [w[0] for w in get_similarity('king', word_to_id, W_in)])
print("Similar to 'france':", [w[0] for w in get_similarity('france', word_to_id, W_in)])
print("Similar to 'apple':", [w[0] for w in get_similarity('apple', word_to_id, W_in)])
print("Similar to 'water':", [w[0] for w in get_similarity('water', word_to_id, W_in)])
print("Similar to 'water':", get_similarity('water', word_to_id, W_in))

Similar to 'king': ['queen', 'elizabeth', 'henry', 'constantine', 'emperor', 'prince', 'frederick', 'vii']
Similar to 'france': ['spain', 'italy', 'austria', 'portugal', 'germany', 'norway', 'denmark', 'netherlands']
Similar to 'apple': ['macintosh', 'ibm', 'ms', 'amiga', 'dos', 'os', 'nintendo', 'pc']
Similar to 'water': ['grain', 'salt', 'fresh', 'dust', 'oxygen', 'clouds', 'ore', 'rain']
Similar to 'king': [('queen', np.float64(0.8441665046832652)), ('elizabeth', np.float64(0.8185578686952146)), ('henry', np.float64(0.814243466727808)), ('constantine', np.float64(0.8053404494779137)), ('emperor', np.float64(0.7998000329166599)), ('prince', np.float64(0.7976997107753362)), ('frederick', np.float64(0.7972661827760088)), ('vii', np.float64(0.7926535578774716))]


### Store Weights and Vocabulary

In [7]:
import pickle

def save_model(W_in, word_to_id, id_to_word, filename="word2vec_model"):
    # 1. Save the weights as a fast binary file
    np.save(f"{filename}_weights.npy", W_in)
    
    # 2. Save the dictionaries using Pickle (standard for Python dicts)
    with open(f"{filename}_vocab.pkl", "wb") as f:
        pickle.dump({'w2i': word_to_id, 'i2w': id_to_word}, f)
    
    print(f"Model saved to {filename}_weights.npy and {filename}_vocab.pkl")

def load_model(filename="word2vec_model"):
    # 1. Load the NumPy array
    W_in = np.load(f"{filename}_weights.npy")
    
    # 2. Load the dictionaries
    with open(f"{filename}_vocab.pkl", "rb") as f:
        vocab = pickle.load(f)
        
    return W_in, vocab['w2i'], vocab['i2w']

# Save the weights and Vocabulary
save_model(W_in, word_to_id, id_to_word)

Model saved to word2vec_model_weights.npy and word2vec_model_vocab.pkl
